# Load ALL data

In [ ]:
%load_ext autoreload
%autoreload 2

from parse_raw_data import SYSTEMS, build_baselines_dict, compute_slowdowns, parse_baselines, parse_concurrent, get_baselines_dataframe

In [ ]:
# SYSTEMS = ['lumi', 'leonardo']
baselines = parse_baselines(SYSTEMS)

In [ ]:
# SYSTEMS = ['intel']
concurrent = parse_concurrent(SYSTEMS)

In [ ]:
baselines_dict = build_baselines_dict(baselines)

In [ ]:
compute_slowdowns(baselines_dict, concurrent)

In [ ]:
print(get_baselines_dataframe(baselines_dict).to_string(index=False))

In [ ]:
print('Loaded concurrent runs:')
for s, c in concurrent.items():
    print(f"{s:<20} -> {len(c)}")

# Plots

In [ ]:

import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import math
from data_types import STRATEGY_ORDER, PLACEMENT_ORDER, SYSTEM_ORDER, SYSTEM_NAMES_MAP, Placement, RunKey, ConcurrentRun
from collections import defaultdict
import os
from typing import Dict, List

## Network Relevance

In [ ]:
data = []
for baseline, m in baselines_dict.items():
    if not m.comm_relevance:
        print('No relevance for')
        print(baseline)
        continue
    data.append({
        "system":       baseline.system,
        "strategy":     baseline.strategy.short(),
        "model":        baseline.model.short(),
        "placement":    baseline.placement_class.value,
        "gpus":         baseline.gpus,
        "comm":         m.comm_relevance.median,
    })
df = pd.DataFrame(data)
df_len = len(df)
df = df[df['comm'] <= 1.0] # filter out wrong alps experiments
print(f'# values over 100%: {df_len-len(df)}')
df["strategy_model"] = df["strategy"] + "\n" + df["model"]
# df[(df['strategy']=="DP+PP") & (df['model']=="llama3-8b") & (df['placement']=="na")]

strategy_order_labels = [s.short() for s in STRATEGY_ORDER]
placement_order_labels = [p.value for p in PLACEMENT_ORDER]

df["strategy"] = pd.Categorical(
    df["strategy"],
    categories=strategy_order_labels,
    ordered=True,
)
df["placement"] = pd.Categorical(
    df["placement"],
    categories=placement_order_labels, 
    ordered=True,
)
strategy_model_order = [
    f"{s.short()}\n{m}"
    for s in STRATEGY_ORDER
    for m in sorted(df["model"].unique())
]
# df["strategy_model"] = pd.Categorical(
#     df["strategy_model"],
#     categories=strategy_model_order,
#     ordered=True,
# )
df = df.sort_values(
    by=["system", "strategy", "model", "placement", "gpus"]
)

In [ ]:
sns.set_theme(palette="muted", style="white", font_scale=1.5) 
mpl.rcParams.update({
    #"text.usetex": True,
    #"text.latex.preamble": r"\usepackage{siunitx} \usepackage{sansmath} \sansmath",
    "font.size": 16,
    "axes.titlesize": 24,
    "axes.titleweight": "bold",
    "axes.labelsize": 24,
    "xtick.labelsize": 20,
    "ytick.labelsize": 22,
    "legend.fontsize": 9,
    "legend.title_fontsize": 12,
    "figure.titlesize": 20,
    "axes.spines.right": False, # Disable top and left spines by default (Tufte style)
    "axes.spines.top": False,
})

In [ ]:
def plot_comm_scatter(
    df: pd.DataFrame,
    output_path: Path,
    nrows: int | None = None,
    ncols: int | None = None,
):
    # ----------------------------
    # Encode x/y positions
    # ----------------------------
    strategies = df["strategy_model"].unique()
    strategy_to_x = {s: i for i, s in enumerate(strategies)}

    placements = df["placement"].unique()
    placement_to_offset = {
        # p: (i - len(placements) / 2) * 0.12 + 0.3 for i, p in enumerate(placements)
        p: 0.0 for i, p in enumerate(placements)
    }
    df = df.copy()
    df["x_base"] = df["strategy_model"].map(strategy_to_x)
    df["x"] = df["x_base"] + df["placement"].map(placement_to_offset).astype(float)
    df["comm_perc"] = df["comm"] * 100.0

    systems = list(df["system"].unique())
    n_systems = len(systems)

    # ----------------------------
    # Layout (custom rows/cols)
    # ----------------------------
    if nrows is None and ncols is None:
        ncols = math.ceil(math.sqrt(n_systems))
        nrows = math.ceil(n_systems / ncols)
    elif nrows is None:
        nrows = math.ceil(n_systems / ncols)
    elif ncols is None:
        ncols = math.ceil(n_systems / nrows)

    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(8 * ncols, 7 * nrows),
        sharex=True,
        sharey=False,
    )

    axes = np.array(axes).reshape(-1)

    cmap = plt.cm.viridis

    systems = [s for s in SYSTEM_ORDER if s in systems]

    AGGREGATE_SYSTEMS = {"jupiter", "leonardo"}

    # ----------------------------
    # Plot
    # ----------------------------
    for ax_i, (ax, system) in enumerate(zip(axes, systems)):
        subdf = df[df["system"] == system]

        # system-specific y mapping
        gpus = sorted(subdf["gpus"].unique())
        gpu_to_y = {g: i for i, g in enumerate(gpus)}

        subdf = subdf.copy()
        subdf["y"] = subdf["gpus"].map(gpu_to_y)
        ax.grid(True, alpha=0.3)

        if system in AGGREGATE_SYSTEMS:
            agg = (
                subdf.groupby(["x_base", "y", "gpus"])["comm_perc"]
                .agg(["min", "max", "mean"])
                .reset_index()
            )
            ax.scatter(
                agg["x_base"],
                agg["y"],
                c=agg["max"],
                cmap=cmap,
                vmin=df["comm_perc"].min(),
                vmax=df["comm_perc"].max(),
                marker="s",  # square marker
                s=1100,
                edgecolor="black",
                linewidth=0.5,
                alpha=0.9,
            )
            for _, row in agg.iterrows():
                if abs(row["min"] - row["max"]) < 1.0:
                    label = f"{row['min']:.0f}"
                    short_label = True
                else:
                    label = f"{row['min']:.0f}-{row['max']:.0f}"
                    short_label = False
                ax.annotate(
                    label,
                    xy=(row["x_base"], row["y"]),
                    ha="center",
                    va="center",
                    fontsize=16 if short_label else 10,
                    fontweight="bold",
                    color="white",
                )
        else:
            for placement, sdf in subdf.groupby("placement"):
                placement_enum = Placement(placement)
                ax.scatter(
                    sdf["x"],
                    sdf["y"],
                    c=sdf["comm_perc"],
                    cmap=cmap,
                    vmin=df["comm_perc"].min(),
                    vmax=df["comm_perc"].max(),
                    marker="s",
                    s=1100,
                    edgecolor="black",
                    linewidth=0.5,
                    alpha=0.9,
                    label=placement_enum.display(new_line=False),
                )
                for _, row in sdf.iterrows():
                    ax.annotate(
                        f"{row['comm_perc']:.0f}",
                        xy=(row["x"], row["y"]),
                        ha="center",
                        va="center",
                        fontsize=16,
                        fontweight="bold",
                        color="white",
                    )

        ax.set_title(SYSTEM_NAMES_MAP[system])
        ax.set_yticks(range(len(gpus)))
        ax.set_yticklabels(gpus)
        if ax_i == 0:
            ax.set_ylabel("GPUs")
        ax.set_xticks(range(len(strategies)))
        ax.set_xticklabels(strategies, ha="center")

    # Remove unused axes
    for ax in axes[n_systems:]:
        ax.remove()

    # ----------------------------
    # Global colorbar (right, outside)
    # ----------------------------
    sm = plt.cm.ScalarMappable(cmap=cmap)
    sm.set_array(df["comm_perc"])

    fig.subplots_adjust(right=0.89)

    cbar_ax = fig.add_axes([0.90, 0.15, 0.01, 0.7])
    cbar = fig.colorbar(sm, cax=cbar_ax)
    cbar.set_label("Communication Time (%)")

    # ----------------------------
    # Global legend (top) - only from non-aggregated systems
    # ----------------------------
    # unique = {}
    # for ax, system in zip(axes[:n_systems], systems):
    #     if system in AGGREGATE_SYSTEMS:
    #         continue
    #     handles, labels = ax.get_legend_handles_labels()
    #     for h, l in zip(handles, labels):
    #         if l not in unique:
    #             unique[l] = h

    # fig.legend(
    #     unique.values(),
    #     unique.keys(),
    #     loc="upper center",
    #     ncol=len(unique),
    #     fontsize=12,
    #     frameon=False,
    # )

    fig.subplots_adjust(top=0.85)

    # ----------------------------
    # Save
    # ----------------------------
    output_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close()

plot_comm_scatter(df, Path('plots/breakdown/comm_relevance'), ncols=7)

## Scaling

In [ ]:
from plot_scaling_and_breakdown import plot_global_faceted
baselines_df = get_baselines_dataframe(baselines_dict)
baselines_df.map(lambda x: f"{x:.1f}" if isinstance(x, float) else x)

In [ ]:
if baselines_df is not None:
    for metric in ['max', 'median', 'geomean']:
        for plot_type in ["scaling"]: #, "efficiency"):
            plot_global_faceted(
                baselines_df,
                output_dir                = Path('plots/scaling'),
                metric                    = metric,
                no_ideal                  = False,
                aggregate_placements_flag = False,
                n_rows                    = 1,
                plot_type                 = plot_type,
                figsize_per_cell          = (6, 7),
            )
else:
    print("No dataframe!!!")

## Slowdowns Analysis

In [ ]:
from plot_slowdown import format_gpus, _sort_run

for system, sys_concurrent in concurrent.items():
    print(f'System: {system}')
    for c in sys_concurrent:
        if system == 'leonardo':
            print(c.job_id)
        if c.job_id == 38409415:
            slowdowns: Dict[RunKey, List[float]] = defaultdict(list)
            for key, measurements in c.multi_runs.items():
                print(key)
                print(baselines_dict[key])
                print(measurements[0]._df)
                print(measurements[0].get_throughput_trimmed_mean())
                print()
                
            for key, s in c.slowdowns.items():
                slowdowns[key].extend(s)
                
            # print(c)
            print(c.display(include_runs=False))
            print(f"{'Category':55s} {'Runs':>5s}  {'Mean':>6s}  {'Median':>6s}  {'Max':>6s}  {'%>1':>5s}")
            print("-" * 93)
            for cat in sorted(slowdowns, key=_sort_run):
                v = np.array(slowdowns[cat])
                lbl = f"{cat.strategy.short()} / {format_gpus(cat.gpus)} / {cat.placement_class} / {cat.model}"
                print(
                    f"{lbl:55s} {len(v):5d}  {v.mean():6.3f}  {np.median(v):6.3f}  {v.max():6.3f}  "
                    f"{100*np.mean(v>1.05):5.1f}%"
                )
            print()

In [ ]:
from statistics import geometric_mean
from typing import Literal, Union


def plot_slowdown_heatmaps(
    concurrent: dict[str, List[ConcurrentRun]],
    output_dir: Path,
    metric: Union[Literal['min'], Literal['max'], Literal['median'], Literal['mean'], Literal['geomean']],
    cap: Union[int, None] = None
) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    
    def sort_y(y):
        id, in_reservation, system, gpus, pattern, strategies, placement_score = y
        return (
            in_reservation,
            gpus,
            len(strategies),
            placement_score,
        )
    
    agg_metric = np.median
    if metric == 'min':
        agg_metric = np.min
    elif metric == 'max':
        agg_metric = np.max
    elif metric == 'mean':
        agg_metric = np.mean
    elif metric == 'geomean':
        agg_metric = geometric_mean
        
    def agg_fn(slowdowns):
        slowdowns = np.array(slowdowns)
        # print(f'before cap: {len(slowdowns)}')
        if cap:
            slowdowns_under_cap = slowdowns[(slowdowns-1.0)*100.0 <= cap]
            if slowdowns_under_cap.shape[0] <= 0:
                print('WARNING: cap filtered out all values, rolling back...')
            else:
                slowdowns = slowdowns_under_cap
        # print(f'after cap: {len(slowdowns)}')
        # print()
        return agg_metric(slowdowns)

    records = []

    # -----------------------------
    # 1. Flatten data
    # -----------------------------
    for system, sys_concurrent in concurrent.items():
        print(f'Plotting system {system}')
        for c in sys_concurrent:
            print('\n  ' + c.display())
            strategies = tuple(sorted(s.short() if hasattr(s, "short") else str(s)
                          for s in c.get_distinct_strategies()))
            y_key = (
                c.job_id,
                c.is_in_reservation(),
                c.system,
                c.gpus,
                c._summarize_pattern(),
                strategies,
                c.get_placement_score(),
            )

            for rk, values in c.slowdowns.items():
                if not values:
                    continue

                records.append({
                    "system": system,
                    "y": y_key,
                    "strategy": rk.strategy.short(),
                    "model": str(rk.model),
                    "gpus": rk.gpus,
                    "x": (rk.strategy.short(), str(rk.model), rk.gpus),
                    f"{metric}_slowdown": (float(agg_fn(values))-1.0)*100.0,
                })

    if not records:
        print("No data available for heatmaps.")
        return

    df = pd.DataFrame(records)

    # -----------------------------
    # 2. Plot per system
    # -----------------------------
    for system in sorted(df["system"].unique()):
        sub = df[df["system"] == system]

        if sub.empty:
            continue

        # -----------------------------
        # 3. Pivot
        # -----------------------------
        pivot = sub.pivot_table(
            index="y",
            columns="x",
            values=f"{metric}_slowdown",
            aggfunc="max",
        )

        if pivot.empty:
            continue

        # pivot = pivot.sort_index(axis=0)
        pivot = pivot.sort_index(axis=1)
        pivot = pivot.sort_index(
            axis=0,
            key=lambda idx: [sort_y(y) for y in idx]
        )

        # -----------------------------
        # 4. Pretty labels
        # -----------------------------
        def fmt_y(y):
            id, in_reservation, sys, gpus, pattern, strategies, placement_score = y
            strat = ','.join(strategies)
            placement_score = f'|  ps={placement_score:.2f}' if placement_score else '' 
            return f"{id} {'(RESV)' if in_reservation else ''} | {gpus}GPU | {pattern} | {strat}{placement_score}"

        pivot.index = [fmt_y(y) for y in pivot.index]
        pivot.columns = [f"{s}\n{m}\n{g}g" for (s, m, g) in pivot.columns]

        # -----------------------------
        # 5. Plot
        # -----------------------------
        plt.figure(figsize=(25, max(12, len(pivot) * 0.45)))

        sns.heatmap(
            pivot,
            annot=True,
            fmt=".0f",
            cmap="viridis",
            linewidths=0.5,
            linecolor="lightgray",
            cbar_kws={"label": f"{metric.capitalize()} Slowdown %"},
        )

        plt.title(f"{metric.capitalize()} Slowdown Heatmap — {system}")
        plt.xlabel("Strategy / Model / GPUs")
        plt.ylabel("Concurrent configuration")
        plt.tight_layout()
        plt.savefig(output_dir / f"{system}_{metric}{'_cap' + str(cap) if cap else ''}_heatmap.png", dpi=300)
        plt.close()
        
for metric in ['max', 'median', 'mean', 'geomean']:
    plot_slowdown_heatmaps(concurrent, Path('plots/heatmaps'), metric, 400)

## Global Summary Slowdown Plots

In [ ]:
from plot_slowdown import format_gpus, _sort_run, plot_slowdown_boxplot, plot_slowdown_violin

CAP = 400

for system, sys_concurrent in concurrent.items():
    # if system != 'leonardo':
    #     continue
    slowdowns_lists: Dict[RunKey, List[float]] = defaultdict(list)
    for c in sys_concurrent:
        for key, s in c.slowdowns.items():
            slowdowns_lists[key].extend(s)
            
    slowdowns: Dict[RunKey, np.ndarray] = {}
    cap_ignored = False
    for k, arr in slowdowns_lists.items():
        len_before = len(arr)
        arr = (np.array(arr)-1.0)*100.0 
        if CAP:
            slowdowns_under_cap = arr[arr <= CAP]
            if slowdowns_under_cap.shape[0] <= 0:
                print('WARNING: cap filtered out all values, rolling back...')
                cap_ignored = True
            else:
                arr = slowdowns_under_cap
        n_excluded = len_before - len(arr)
        if n_excluded > 0:
            print(f'CAP excluded {n_excluded} values for {k.display()}')
        slowdowns[k] = arr
            
    print(f'System: {system}')
    print(f"{'Run Type':50s} {'Runs':>5s}  {'Mean':>8s}  {'Median':>8s}  {'%>5':>5s}")
    print("-" * 85)
    for cat in sorted(slowdowns, key=_sort_run):
        v = slowdowns[cat]
        lbl = f"{cat.strategy.short()} / {format_gpus(cat.gpus)} / {cat.placement_class} / {cat.model}"
        print(
            f"{lbl:50s} {len(v):5d}  {v.mean():8.3f}  {np.median(v):8.3f}  "
            f"{100*np.mean(v>5):5.1f}%"
        )

    output_dir = 'plots/slowdown_all'
    os.makedirs(output_dir, exist_ok=True)
    base_path = os.path.join(output_dir, f"slowdown_{system}.png")

    # plot_slowdown_violin(slowdowns, system, base_path.replace(".png", "_violin.png"))

    # Determine a y-clip that keeps all box upper-whiskers visible.
    max_whisker = 1.0
    for v in slowdowns.values():
        arr = v
        q1, q3 = float(np.percentile(arr, 25)), float(np.percentile(arr, 75))
        whisker_hi = min(float(arr.max()), q3 + 1.5 * (q3 - q1))
        max_whisker = max(max_whisker, whisker_hi)
    boxplot_y_clip = max(1.2, round(max_whisker * 1.1, 1))

    plot_slowdown_boxplot(slowdowns, system, base_path.replace(".png", f"_cap{'NO' if cap_ignored else CAP}_boxplot.png"), y_clip=boxplot_y_clip)
    # plot_slowdown_strip(slowdowns, system, base_path.replace(".png", "_strip.png"))
    # plot_slowdown_boxplot_stacked(slowdowns, base_path.replace(".png", "_boxplot_stacked.png"))
    print()
    print()